In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
repos_root = Path.cwd().parent
sys.path.append(str(repos_root))

from protos import packet_pb2, ripple_pb2
from rocket_controller.encoder_decoder import PacketEncoderDecoder, DecodingNotSupportedError


In [ ]:
log_dir = "/Users/lli21/rocket/logs/2025_11_27_15h46m/G0T1/iteration-1"
action_file = f"{log_dir}/action-1.csv"
subscriber_file = f"{log_dir}/subscribe_event_log.csv"

df_action = pd.read_csv(action_file)
df_subscriber = pd.read_csv(subscriber_file)
df_subscriber.head()

df_TMValidation = df_action[df_action["message_type"] == "TMValidation"]
df_TMValidation.head()

def packet_data_to_dict(packet_data):
        hex_str = packet_data.strip()
        packet_bytes = bytes.fromhex(hex_str)
        pkt = packet_pb2.Packet()
        pkt.data = packet_bytes
        message, message_type = PacketEncoderDecoder.decode_packet(pkt)
        validation_dict = PacketEncoderDecoder.decode_validation(message)
        return validation_dict

df_TMValidation["packet_data"] = df_TMValidation["packet_data"].map(packet_data_to_dict)

df_TMValidation['ledger_sequence'] = df_TMValidation["packet_data"].map(
    lambda x: x.get('LedgerSequence')
)

# 2. 根据新列筛选
df_TMValidation = df_TMValidation[df_TMValidation['ledger_sequence'] <= 15]
end_point = df_TMValidation.iloc[-1].timestamp
end_point

df_action = df_action[df_action["timestamp"] <= end_point]


In [ ]:
nodes = sorted(df_action["from_node_id"].dropna().unique().astype(int).tolist())
node_to_y = {n: i for i, n in enumerate(nodes)}

start_time = df_action["timestamp"].min()
end_time = df_action["timestamp"].max()
sec_elapsed = (end_time - start_time) / 1000
print(f"time elapsed: {sec_elapsed:.2f} seconds, = {sec_elapsed / 60:.2f} minutes")
client_y = len(nodes)

In [ ]:
def to_sec(t):
    return (t - start_time) / 1000

In [ ]:
n_lanes = len(nodes) + 1
fig, ax = plt.subplots(figsize=(10, max(2, n_lanes * 0.5)))
ax.hlines(client_y, xmin=to_sec(start_time), xmax=to_sec(end_time), color="#eeeeee")
ax.text(to_sec(start_time) - 0.01 * (to_sec(end_time)-to_sec(start_time)), client_y, 'C', va="center", ha="right", fontsize=8)
for n, y in node_to_y.items():
    ax.hlines(y, xmin=to_sec(start_time), xmax=to_sec(end_time), color="#dddddd")
    ax.text(to_sec(start_time) - 0.01 * (to_sec(end_time)-to_sec(start_time)), y, str(n), va="center", ha="right", fontsize=8)
ax.set_ylim(-1, n_lanes)

In [ ]:
    ######### 画 validation 消息
# 创建颜色映射
import matplotlib.cm as cm
ledger_sequences = df_TMValidation['ledger_sequence'].dropna().unique()
ledger_sequences = sorted([seq for seq in ledger_sequences if 1 <= seq <= 15])
colors = cm.viridis(np.linspace(0, 1, len(ledger_sequences)))  # 或者使用其他colormap
seq_to_color = dict(zip(ledger_sequences, colors))

MAX_U32 = 2**32 - 1
for _, row in df_TMValidation.iterrows():
    try:
        frm = int(row["from_node_id"])
        to = int(row["to_node_id"])
    except Exception:
        continue
    t = row["timestamp"]
    delay = int(row["action"])
    
    if delay == MAX_U32:
        # skip drops
        continue
    
    seq = row["packet_data"].get("LedgerSequence", None) # 1-15
    
    

    
    t2 = t + delay  
    x1 = to_sec(t)
    x2 = to_sec(t2)
    y1 = node_to_y.get(frm, None)
    y2 = node_to_y.get(to, None)
    if y1 is None or y2 is None:
        continue
    # color by whether delayed
    color = seq_to_color.get(seq)
    # draw arrow
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color=color, lw=0.8))
    
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=seq_to_color[seq], label=f'{seq}') 
                   for seq in ledger_sequences]
ax.legend(handles=legend_elements, loc='best', title="LedgerSeq", fontsize=8)

from IPython.display import display
display(fig)
